##### Ideal QSVM (SVC Precomputed Kernel) - Spambase - PCA

In [ ]:
!pip install qiskit qiskit-machine-learning qiskit-aer seaborn

In [ ]:
import qiskit, qiskit_aer, qiskit_machine_learning
print("Qiskit:", qiskit.__version__)
print("Aer:", qiskit_aer.__version__)
print("QML:", qiskit_machine_learning.__version__)

In [ ]:
# --- Import Libraries ---
import pandas as pd
import numpy as np
import time
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, recall_score, balanced_accuracy_score, precision_score, f1_score

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

In [ ]:
# --- Qiskit Imports ---
from qiskit.circuit.library import ZZFeatureMap
from qiskit_aer import AerSimulator
from qiskit.primitives import StatevectorSampler as Sampler
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel

##### Load Dataset

In [ ]:
# --- Import Spambase Column Names ---
spambase_columns = [
    "word_freq_make", "word_freq_address", "word_freq_all", "word_freq_3d",
    "word_freq_our", "word_freq_over", "word_freq_remove", "word_freq_internet",
    "word_freq_order", "word_freq_mail", "word_freq_receive", "word_freq_will",
    "word_freq_people", "word_freq_report", "word_freq_addresses", "word_freq_free",
    "word_freq_business", "word_freq_email", "word_freq_you", "word_freq_credit",
    "word_freq_your", "word_freq_font", "word_freq_000", "word_freq_money",
    "word_freq_hp", "word_freq_hpl", "word_freq_george", "word_freq_650",
    "word_freq_lab", "word_freq_labs", "word_freq_telnet", "word_freq_857",
    "word_freq_data", "word_freq_415", "word_freq_85", "word_freq_technology",
    "word_freq_1999", "word_freq_parts", "word_freq_pm", "word_freq_direct",
    "word_freq_cs", "word_freq_meeting", "word_freq_original", "word_freq_project",
    "word_freq_re", "word_freq_edu", "word_freq_table", "word_freq_conference",
    "char_freq_;", "char_freq_(", "char_freq_[", "char_freq_!",
    "char_freq_$", "char_freq_#", "capital_run_length_average",
    "capital_run_length_longest", "capital_run_length_total", "label"
]

# --- Load the Spambase Dataset ---
file_path = '/kaggle/input/spambase/spambase.data'
# file_path = r'C:\Users\User\Documents\MyProjects\FYP_ResearchProject\data\spambase\spambase.data'
df = pd.read_csv(file_path, header=None, names=spambase_columns)
df.drop_duplicates(inplace=True)

print(f"Dataset loaded: {df.shape[0]} samples, {df.shape[1]} features")

##### Experiment Configurations

In [ ]:
# ==========================================
# EXPERIMENT CONFIGURATIONS (PCA)
# ==========================================

experiments = [
    # EXP 1: Shot Noise Effect
    {'id': 'Exp1_128shots', 'samples': 700, 'n_components': 7, 'shots': 128, 'reps': 1, 'entanglement': 'linear'},
    {'id': 'Exp1_512shots', 'samples': 700, 'n_components': 7, 'shots': 512, 'reps': 1, 'entanglement': 'linear'},
    {'id': 'Exp1_1024shots', 'samples': 700, 'n_components': 7, 'shots': 1024, 'reps': 1, 'entanglement': 'linear'},
    
    # EXP 2: Reps Effect
    {'id': 'Exp2_Reps1', 'samples': 700, 'n_components': 7, 'shots': 1024, 'reps': 1, 'entanglement': 'linear'},
    {'id': 'Exp2_Reps2', 'samples': 700, 'n_components': 7, 'shots': 1024, 'reps': 2, 'entanglement': 'linear'},
    
    # EXP 3: Entanglement
    {'id': 'Exp3_Circular', 'samples': 700, 'n_components': 7, 'shots': 1024, 'reps': 1, 'entanglement': 'circular'},
    {'id': 'Exp3_Full', 'samples': 700, 'n_components': 7, 'shots': 1024, 'reps': 1, 'entanglement': 'full'},
]

print(f"Total experiments configured: {len(experiments)}")
print("Running might take hours or even several days")

##### Main Experiment

In [ ]:
# ==========================================
# MAIN EXPERIMENT LOOP
# ==========================================

all_results = []
X = df.drop('label', axis=1)
y = df['label']

for exp_num, config in enumerate(experiments, 1):
    print("\n" + "=" * 80)
    print(f"EXPERIMENT {exp_num}/{len(experiments)}: {config['id']}")
    print("=" * 80)
    print(f"  Samples: {config['samples']}")
    print(f"  PCA Components: {config['n_components']}")
    print(f"  Shots: {config['shots']}")
    print(f"  Reps: {config['reps']}")
    print(f"  Entanglement: {config['entanglement']}")
    print("=" * 80)

    # === 1. CREATE DATASET ===
    train_samples = config['samples']
    subset_size = int(round(train_samples / 0.7))

    X_subset, _, y_subset, _ = train_test_split(
        X, y, train_size=subset_size, stratify=y, random_state=42
    )

    X_train, X_test, y_train, y_test = train_test_split(
        X_subset, y_subset, test_size=0.30, random_state=42, stratify=y_subset
    )

    print(f"\nDataset created: {X_train.shape[0]} train, {X_test.shape[0]} test")

    # === 2. VARIANCE FILTERING & SCALING ===
    selector_variance = VarianceThreshold(threshold=0)
    X_train_filtered = selector_variance.fit_transform(X_train)
    X_test_filtered = selector_variance.transform(X_test)
    remaining_cols = X_train.columns[selector_variance.get_support()]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_filtered)
    X_test_scaled = scaler.transform(X_test_filtered)
    X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=remaining_cols)
    X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=remaining_cols)

    print("Data scaled successfully")

    # === 3. CORRELATION-BASED FEATURE DROPPING ===
    THRESH = 0.9
    corr_matrix_train = X_train_scaled_df.corr().abs()
    upper_triangle = corr_matrix_train.where(
        np.triu(np.ones(corr_matrix_train.shape), k=1).astype(bool)
    )

    columns_to_drop = set()
    for column in upper_triangle.columns:
        high_corr_partners = upper_triangle.index[upper_triangle[column] > THRESH].tolist()
        if high_corr_partners:
            for partner in high_corr_partners:
                corr_main_vs_target = y_train.reset_index(drop=True).corr(pd.Series(X_train_scaled_df[column].values))
                corr_partner_vs_target = y_train.reset_index(drop=True).corr(pd.Series(X_train_scaled_df[partner].values))

                if abs(corr_main_vs_target) < abs(corr_partner_vs_target):
                    columns_to_drop.add(column)
                else:
                    columns_to_drop.add(partner)

    to_drop_final = sorted(list(columns_to_drop))
    X_train_selected = X_train_scaled_df.drop(columns=to_drop_final)
    X_test_selected = X_test_scaled_df.drop(columns=to_drop_final)

    print(f"Dropped {len(to_drop_final)} highly correlated features")

    # === 4. PCA DIMENSIONALITY REDUCTION ===
    n_components = config['n_components']
    max_components = min(n_components, X_train_selected.shape[1], X_train_selected.shape[0])
    
    pca = PCA(n_components=max_components, random_state=42)
    X_train_pca = pca.fit_transform(X_train_selected)
    X_test_pca = pca.transform(X_test_selected)
    
    explained_variance = sum(pca.explained_variance_ratio_) * 100
    print(f"PCA: Reduced to {max_components} components (Explained variance: {explained_variance:.2f}%)")

    # === 5. QUANTUM KERNEL SETUP ===
    fm = ZZFeatureMap(
        feature_dimension=max_components,
        reps=config['reps'],
        entanglement=config['entanglement']
    )

    sampler = Sampler(default_shots=config['shots'])
    fidelity = ComputeUncompute(sampler=sampler)
    qkernel = FidelityQuantumKernel(fidelity=fidelity, feature_map=fm)

    print(f"Quantum kernel configured (ZZFeatureMap, reps={config['reps']}, entanglement={config['entanglement']})")

    # === 6. COMPUTE KERNEL MATRICES ===
    print("\nComputing kernel matrices...")
    start_kernel = time.time()

    matrix_train = qkernel.evaluate(x_vec=X_train_pca)
    matrix_test = qkernel.evaluate(x_vec=X_test_pca, y_vec=X_train_pca)

    kernel_time = time.time() - start_kernel
    print(f"Kernel computation: {kernel_time:.2f}s")

    # === 7. TRAIN MODEL ===
    print("\nUsing optimal C...")
    best_c = 100.0
    print(f"  → Fixed C: {best_c} (from classical RBF)")

    start_train = time.time()
    best_model = SVC(kernel='precomputed', C=best_c, class_weight='balanced', random_state=42)
    best_model.fit(matrix_train, y_train)
    train_time = time.time() - start_train

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    cv_scores = cross_val_score(best_model, matrix_train, y_train, cv=cv, scoring='accuracy')
    cv_score = cv_scores.mean()

    print(f"  → CV Score: {cv_score:.4f}")
    print(f"  → Training time: {train_time:.2f}s")

    # === 8. EVALUATION ===
    y_train_pred = best_model.predict(matrix_train)
    y_test_pred = best_model.predict(matrix_test)

    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    test_balanced_acc = balanced_accuracy_score(y_test, y_test_pred)
    precision = precision_score(y_test, y_test_pred, pos_label=1, zero_division=0)
    recall = recall_score(y_test, y_test_pred, pos_label=1, zero_division=0)
    f1 = f1_score(y_test, y_test_pred, pos_label=1, zero_division=0)
    gen_gap = abs(train_acc - test_acc)

    print(f"  → Train Accuracy: {train_acc:.4f}")
    print(f"  → Test Accuracy: {test_acc:.4f}")
    print(f"  → Test Balanced Accuracy: {test_balanced_acc:.4f}")
    print(f"  → Precision: {precision:.4f}")
    print(f"  → Recall: {recall:.4f}")
    print(f"  → F1-Score: {f1:.4f}")
    print(f"  → Generalization Gap: {gen_gap:.4f}")

    # === 9. STORE RESULTS ===
    all_results.append({
        'experiment_id': config['id'],
        'exp_number': exp_num,
        'samples': config['samples'],
        'n_components': max_components,
        'explained_variance': explained_variance,
        'shots': config['shots'],
        'reps': config['reps'],
        'entanglement': config['entanglement'],
        'best_c': best_c,
        'cv_score': cv_score,
        'train_acc': train_acc,
        'test_acc': test_acc,
        'test_balanced_acc': test_balanced_acc,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'gen_gap': gen_gap,
        'kernel_time': kernel_time,
        'train_time': train_time,
        'y_test': y_test.tolist(),
        'y_pred': y_test_pred.tolist()
    })

    print("\n" + "Classification Report:")
    print(classification_report(y_test, y_test_pred, zero_division=0))

    np.save(f'kernel_train_{config["id"]}.npy', matrix_train)
    np.save(f'kernel_test_{config["id"]}.npy', matrix_test)
    print(f"Saved kernel matrices: kernel_train_{config['id']}.npy, kernel_test_{config['id']}.npy")

print("\n" + "=" * 80)
print("ALL EXPERIMENTS COMPLETE!")
print("=" * 80)

In [ ]:
# === SAVE RESULTS ===
results_df = pd.DataFrame(all_results)
results_df.to_csv('qsvm_pca_all_experiments.csv', index=False)

print("\n" + "=" * 80)
print("RESULTS SUMMARY")
print("=" * 80)
print(results_df[['experiment_id', 'samples', 'n_components', 'shots', 'reps', 'entanglement', 
                   'test_acc', 'test_balanced_acc', 'spam_recall', 'gen_gap', 'kernel_time']])
print(f"\nFull results saved to: qsvm_pca_all_experiments.csv")

In [ ]:
# === CONFUSION MATRICES ===
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

print("\nGenerating confusion matrices for all experiments...")

fig_rows = (len(experiments) + 3) // 4
fig, axes = plt.subplots(fig_rows, 4, figsize=(20, 5 * fig_rows))
axes = axes.flatten()

for idx, result in enumerate(all_results):
    y_test = np.array(result['y_test'])
    y_pred = np.array(result['y_pred'])
    
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Non-Spam', 'Spam'])
    
    disp.plot(ax=axes[idx], cmap='Blues', values_format='d')
    axes[idx].set_title(f"{result['experiment_id']}\nAcc: {result['test_acc']:.3f}", 
                        fontsize=10, fontweight='bold')
    axes[idx].grid(False)

for idx in range(len(all_results), len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig('confusion_matrices_pca_experiments.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Confusion matrices saved: confusion_matrices_pca_experiments.png")

In [ ]:
# === BEST CONFIGURATIONS ANALYSIS ===
print("\n" + "=" * 80)
print("BEST CONFIGURATIONS ANALYSIS")
print("=" * 80)

# Best Test Accuracy
best_acc_idx = results_df['test_acc'].idxmax()
best_acc = results_df.iloc[best_acc_idx]
print(f"\nBEST TEST ACCURACY:")
print(f"  Experiment: {best_acc['experiment_id']}")
print(f"  Test Acc: {best_acc['test_acc']:.4f}, Precision: {best_acc['precision']:.4f}")
print(f"  Test Acc: {best_acc['test_acc']:.4f}, Recall: {best_acc['recall']:.4f}")
print(f"  Test Acc: {best_acc['test_acc']:.4f}, F1-Score: {best_acc['f1']:.4f}")
print(f"  Config: {int(best_acc['n_components'])} PCA components, {int(best_acc['shots'])} shots, reps={int(best_acc['reps'])}, {best_acc['entanglement']}")

# Best Generalization
best_gen_idx = results_df['gen_gap'].idxmin()
best_gen = results_df.iloc[best_gen_idx]
print(f"\nBEST GENERALIZATION:")
print(f"  Experiment: {best_gen['experiment_id']}")
print(f"  Gen Gap: {best_gen['gen_gap']:.4f}, Test Acc: {best_gen['test_acc']:.4f}")

# Fastest
fastest_idx = results_df['kernel_time'].idxmin()
fastest = results_df.iloc[fastest_idx]
print(f"\nFASTEST COMPUTATION:")
print(f"  Experiment: {fastest['experiment_id']}")
print(f"  Kernel Time: {fastest['kernel_time']:.2f}s, Test Acc: {fastest['test_acc']:.4f}")

# Save config
import json
qsvm_config = {
    'best_accuracy': {'experiment_id': best_acc['experiment_id'], 'test_acc': float(best_acc['test_acc'])},
    'best_generalization': {'experiment_id': best_gen['experiment_id'], 'gen_gap': float(best_gen['gen_gap'])},
    'fastest': {'experiment_id': fastest['experiment_id'], 'kernel_time': float(fastest['kernel_time'])}
}
with open('ideal_qsvm_pca_best_configurations.json', 'w') as f:
    json.dump(qsvm_config, f, indent=4)
print("\n✓ Configuration saved to: ideal_qsvm_pca_best_configurations.json")

In [ ]:
# === VISUALIZATION ===
sns.set_style('whitegrid')

# Performance Heatmap
plt.figure(figsize=(12, 8))
metrics = ['test_acc', 'test_balanced_acc', 'spam_recall', 'gen_gap', 'cv_score']
heatmap_data = results_df.set_index('experiment_id')[metrics].T

sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='RdYlGn', 
            vmin=0, vmax=1, linewidths=1, linecolor='white',
            cbar_kws={'label': 'Score'}, 
            annot_kws={'fontsize': 10, 'fontweight': 'bold'})

plt.title('Performance Metrics Across All PCA Experiments', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Experiment ID', fontsize=13, fontweight='bold')
plt.ylabel('Metrics', fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('qsvm_pca_heatmap_summary.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: qsvm_pca_heatmap_summary.png")